In [ ]:
import os
import sys
from pathlib import Path

# Get the notebook's directory
NOTEBOOK_DIR = Path(os.getcwd())
BACKEND_DIR = NOTEBOOK_DIR.parent.parent / "Skin_Lesion_Classification_backend"
ML_DIR = BACKEND_DIR / "ml"

# Add backend ml/ to sys.path so imports like `from src.data.dataset` work
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Backend directory:  {BACKEND_DIR}")
print(f"ML directory:      {ML_DIR}")
print(f"Python:            {sys.executable}")

# Verify backend exists
if not BACKEND_DIR.exists():
    print(f"\u274c ERROR: Backend not found at {BACKEND_DIR}")
    print("Make sure the backend repo is cloned at the same level as the frontend repo.")
else:
    print("\u2705 Backend directory found.")

# RQ6 — External Validation: Distribution Shift Across Populations
## XAI Skin Lesion Classification — Research Question 6

**Research Question**: How do AI models trained on standard archives perform when
subject to external validation on geographically and demographically diverse populations?

**Why This Matters**:
HAM10000 (and ISIC) were collected from specific clinical settings with particular
patient demographics. Real-world deployment会遇到完全不同地理和人口统计学特征的患者.
A model that performs well on HAM10000 may degrade significantly on populations
with different skin tones, lesion presentations, or imaging protocols.

**The Core Problem**: Distribution shift.
Training on one population and deploying on another exposes the model to:
- Different demographic distributions (age, sex, skin tone)
- Different lesion type prevalence
- Different imaging protocols and equipment
- Different lighting conditions

This RQ measures HOW and WHY performance degrades.

---

## CONCEPT: Types of Distribution Shift

**Covariate Shift**: The input distribution changes (different demographics, imaging)
but the mapping from input to label remains the same.

**Prior Probability Shift**: The prevalence of classes changes (e.g., more/fewer
malignant cases in one population vs another).

**Concept Drift**: The relationship between features and labels itself changes
over time or across populations.

**In our context**: All three likely contribute. HAM10000 is predominantly
fair-skinned populations (Australia/USA). Deploying in other regions may expose
the model to different skin tone distributions, lesion presentations, and lighting.

**Key insight**: Performance drop may be accompanied by:
- Lower confidence (model is less certain on OOD inputs)
- Higher inter-method disagreement (RQ4 insight — disagreement signals uncertainty)
- Different CAM focus patterns (attention shifts to background or artifacts)
- Broader/diffuse CAMs (model unable to find coherent lesion signal)

---

## CELL 1: Understanding the External Datasets

**Datasets for External Validation**:

| Dataset | Images | Geography | Demographics | Use for |
|---------|--------|-----------|-------------|---------|
| HAM10000 | 10,015 | Australia/USA | Predominantly fair skin | Training (internal) |
| ISIC 2020 | 33,126 | Global | More diverse | External validation |
| PAD-UFES-20 | 22,000 | Brazil | Mixed skin tones | External validation |
| derm101 | 9,000+ | Mixed | Mixed | External validation |

For this notebook, we use **ISIC 2020** as the external validation set.
It has different characteristics from HAM10000:
- More diverse geographic origin
- Different lesion type distribution
- Different image resolution and quality

**How to download ISIC 2020**:
```bash
kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
# Or from: https://challenge.isic-archive.com/data/2020
```

In [ ]:
import sys
sys.path.append('../Skin_Lesion_Classification_backend/ml')

import pandas as pd
import numpy as np
from pathlib import Path

# Internal training data (HAM10000)
internal_df = pd.read_csv(
    '../Skin_Lesion_Classification_backend/ml/data/processed/metadata_with_paths.csv'
)

# External validation data (ISIC 2020 — download separately)
# Place ISIC 2020 metadata at this path:
external_path = Path('../Skin_Lesion_Classification_backend/ml/data/processed/isic2020_metadata.csv')

if external_path.exists():
    external_df = pd.read_csv(external_path)
    print(f"External dataset loaded: {len(external_df)} images")
else:
    print(f"\u26a0\ufe0f External dataset not found at {external_path}")
    print("Download ISIC 2020 and place metadata CSV at that path.")
    print("\nTo proceed, you can also use a held-out subset of HAM10000 as a proxy.")
    external_df = None

print(f"\nInternal dataset (HAM10000): {len(internal_df)} images")
print(f"Demographics info available: {set(['age', 'sex', 'localization']) & set(internal_df.columns)}")

---

## CELL 2: Compare Internal vs External Dataset Characteristics

**WHY**: Before measuring performance drop, understand HOW the datasets differ.
This helps explain WHY performance drops.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Lesion type distribution
axes[0].set_title('Lesion Type Distribution\n(Internal vs External)')
internal_counts = internal_df['dx'].value_counts()
if external_df is not None:
    external_counts = external_df['diagnosis'].value_counts()
    # Align categories
    all_dx = sorted(set(internal_counts.index) | set(external_counts.index))
    internal_vals = [internal_counts.get(d, 0) for d in all_dx]
    external_vals = [external_counts.get(d, 0) for d in all_dx]
    
    x = np.arange(len(all_dx))
    width = 0.35
    axes[0].bar(x - width/2, internal_vals, width, label='Internal (HAM10000)', color='steelblue')
    axes[0].bar(x + width/2, external_vals, width, label='External (ISIC 2020)', color='coral')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(all_dx, rotation=45, ha='right')
    axes[0].legend()
else:
    internal_counts.plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('Internal (HAM10000)\nNo external data loaded')
    axes[0].tick_params(axis='x', rotation=45)

# Binary label distribution
axes[1].set_title('Binary Label Distribution\n(Benign vs Malignant)')
internal_binary = internal_df['label_name'].value_counts()
internal_binary.plot(kind='bar', ax=axes[1], color=['green', 'red'], alpha=0.7)
if external_df is not None:
    # Map external to benign/malignant if needed
    ext_binary = external_df['diagnosis'].apply(
        lambda x: 'benign' if x in ['nv', 'bkl', 'df', 'vasc'] else 'malignant'
    ).value_counts()
    print(f"External binary: {ext_binary.to_dict()}")
axes[1].tick_params(axis='x', rotation=0)

# Image size/resolution if available
axes[2].set_title('Dataset Size Comparison')
sizes = [len(internal_df)]
labels = ['Internal\n(HAM10000)']
colors = ['steelblue']
if external_df is not None:
    sizes.append(len(external_df))
    labels.append('External\n(ISIC 2020)')
    colors.append('coral')
axes[2].bar(labels, sizes, color=colors)
for i, s in enumerate(sizes):
    axes[2].text(i, s + 200, str(s), ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/figures/RQ6_dataset_comparison.png', dpi=150)
plt.show()

---

## CELL 3: Load Model and Evaluate on Internal vs External Test Sets

**WHY**: Measure HOW performance degrades — AUC, accuracy, confidence differences.

In [ ]:
import sys
sys.path.append('../Skin_Lesion_Classification_backend/ml')

import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

from src.models.classifier import create_model, get_target_layer
from src.data.dataset import get_transforms, create_splits
from sklearn.metrics import roc_auc_score, accuracy_score, precision_recall_fscore_support

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Load model
model = create_model('resnet50', num_classes=2).to(device)
model.load_state_dict(torch.load(
    '../Skin_Lesion_Classification_backend/ml/outputs/models/resnet50_best.pth',
    map_location=device
))
model.eval()
target_layer = get_target_layer(model, 'resnet50')
print("Model loaded.")

---

## CELL 4: Evaluate on Internal Test Set (Baseline)

In [ ]:
from src.data.dataset import create_splits

transform = get_transforms('test', 224)

# Internal evaluation
internal_results = []

print("Evaluating on internal test set (HAM10000)...")
internal_test = internal_df.sample(min(200, len(internal_df)), random_state=42)

for _, row in tqdm(internal_test.iterrows(), total=len(internal_test)):
    try:
        orig = np.array(Image.open(row['filepath']).convert('RGB').resize((224, 224)))
        input_t = transform(image=orig)['image'].unsqueeze(0).to(device)

        with torch.no_grad():
            out = model(input_t)
            probs = torch.softmax(out, dim=1)[0]
            pred = probs.argmax().item()
            conf = probs[pred].item()
            mal_prob = probs[1].item()

        internal_results.append({
            'image_id': row['image_id'],
            'true_label': int(row['label']),
            'pred': pred,
            'confidence': conf,
            'malignant_prob': mal_prob,
            'correct': int(pred == int(row['label']))
        })
    except Exception as e:
        pass

internal_results_df = pd.DataFrame(internal_results)

# Metrics
int_auc = roc_auc_score(internal_results_df['true_label'], internal_results_df['malignant_prob'])
int_acc = accuracy_score(internal_results_df['true_label'], internal_results_df['pred'])
int_avg_conf = internal_results_df['confidence'].mean()

print(f"\n=== INTERNAL (HAM10000) Results ===")
print(f"AUC:        {int_auc:.4f}")
print(f"Accuracy:   {int_acc:.4f}")
print(f"Avg conf:   {int_avg_conf:.4f}")

---

## CELL 5: Evaluate on External Validation Set

**If ISIC 2020 is not available**, use a held-out portion of HAM10000 as a proxy
for demonstrating the methodology. In practice, use real external data for results.

In [ ]:
if external_df is not None:
    print("Evaluating on external dataset (ISIC 2020)...")
    external_results = []
    
    # Sample for efficiency
    external_sample = external_df.sample(min(200, len(external_df)), random_state=42)
    
    for _, row in tqdm(external_sample.iterrows(), total=len(external_sample)):
        try:
            # Adjust path based on your ISIC data location
            img_path = Path(row.get('filepath', row.get('image_path', None)))
            if not img_path.exists():
                img_path = Path(f'../Skin_Lesion_Classification_backend/ml/data/raw/isic2020/{row["image_id"]}.jpg')
            
            orig = np.array(Image.open(img_path).convert('RGB').resize((224, 224)))
            input_t = transform(image=orig)['image'].unsqueeze(0).to(device)

            with torch.no_grad():
                out = model(input_t)
                probs = torch.softmax(out, dim=1)[0]
                pred = probs.argmax().item()
                conf = probs[pred].item()
                mal_prob = probs[1].item()

            # Map diagnosis to binary label (adjust based on your external dataset columns)
            diagnosis = row.get('diagnosis', row.get('dx', 'unknown'))
            true_label = 1 if diagnosis in ['mel', 'bcc'] else 0  # melanoma, bcc = malignant
            
            external_results.append({
                'image_id': row['image_id'],
                'true_label': true_label,
                'pred': pred,
                'confidence': conf,
                'malignant_prob': mal_prob,
                'correct': int(pred == true_label),
                'diagnosis': diagnosis
            })
        except Exception as e:
            pass
    
    external_results_df = pd.DataFrame(external_results)
    
    ext_auc = roc_auc_score(external_results_df['true_label'], external_results_df['malignant_prob'])
    ext_acc = accuracy_score(external_results_df['true_label'], external_results_df['pred'])
    ext_avg_conf = external_results_df['confidence'].mean()
    
    print(f"\n=== EXTERNAL (ISIC 2020) Results ===")
    print(f"AUC:        {ext_auc:.4f}")
    print(f"Accuracy:   {ext_acc:.4f}")
    print(f"Avg conf:   {ext_avg_conf:.4f}")
    print(f"Images evaluated: {len(external_results_df)}")
else:
    # Use a different subset of HAM10000 as OOD proxy
    print("No external data — using held-out HAM10000 subset as OOD proxy...")
    print("NOTE: This is a proxy. Real external validation requires different dataset.")
    
    # Split HAM10000 differently: use lesion types NOT in training as OOD
    # e.g., if training had certain types, hold out others
    print("\nUsing melanoma-only subset as OOD proxy...")
    ood_subset = internal_df[internal_df['dx'] == 'mel'].sample(50, random_state=99)
    
    ood_results = []
    for _, row in tqdm(ood_subset.iterrows(), total=len(ood_subset)):
        try:
            orig = np.array(Image.open(row['filepath']).convert('RGB').resize((224, 224)))
            input_t = transform(image=orig)['image'].unsqueeze(0).to(device)

            with torch.no_grad():
                out = model(input_t)
                probs = torch.softmax(out, dim=1)[0]
                pred = probs.argmax().item()
                conf = probs[pred].item()
                mal_prob = probs[1].item()

            ood_results.append({
                'image_id': row['image_id'],
                'true_label': int(row['label']),
                'pred': pred,
                'confidence': conf,
                'malignant_prob': mal_prob,
                'correct': int(pred == int(row['label'])),
                'dx': row['dx']
            })
        except:
            pass
    
    external_results_df = pd.DataFrame(ood_results)
    ext_auc = roc_auc_score(external_results_df['true_label'], external_results_df['malignant_prob'])
    ext_acc = accuracy_score(external_results_df['true_label'], external_results_df['pred'])
    ext_avg_conf = external_results_df['confidence'].mean()
    
    print(f"\n=== OOD Proxy (Melanoma subset) Results ===")
    print(f"AUC:        {ext_auc:.4f}")
    print(f"Accuracy:   {ext_acc:.4f}")
    print(f"Avg conf:   {ext_avg_conf:.4f}")

---

## CELL 6: Quantify Performance Drop

In [ ]:
# Performance drop summary
print("=" * 50)
print("PERFORMANCE DROP ANALYSIS")
print("=" * 50)
print()
print(f"{'Metric':<20} {'Internal':>12} {'External':>12} {'Drop':>12}")
print("-" * 58)
print(f"{'AUC':<20} {int_auc:>12.4f} {ext_auc:>12.4f} {int_auc - ext_auc:>12.4f}")
print(f"{'Accuracy':<20} {int_acc:>12.4f} {ext_acc:>12.4f} {int_acc - ext_acc:>12.4f}")
print(f"{'Avg Confidence':<20} {int_avg_conf:>12.4f} {ext_avg_conf:>12.4f} {int_avg_conf - ext_avg_conf:>12.4f}")

print()
auc_drop = int_auc - ext_auc
acc_drop = int_acc - ext_acc
conf_drop = int_avg_conf - ext_avg_conf

print("INTERPRETATION:")
if auc_drop > 0.05:
    print(f"  \u274c Significant AUC drop ({auc_drop:.3f}) — model struggles on external data")
elif auc_drop > 0.01:
    print(f"  \u26a0\ufe0f Moderate AUC drop ({auc_drop:.3f}) — some generalization degradation")
else:
    print(f"  \u2705 Minimal AUC drop ({auc_drop:.3f}) — model generalizes well")

if conf_drop > 0.05:
    print(f"  \u2705 External predictions are less confident ({conf_drop:.3f}) — model recognizes uncertainty")
else:
    print(f"  \u274c External predictions nearly as confident — model may be overconfident on OOD data")

---

## CELL 7: XAI Analysis — Do CAMs Change Under Distribution Shift?

**WHY**: Performance drop may be accompanied by changed attention patterns.
If the model highlights different regions on external data, it may be
relying on different features (potentially artifacts or biases).

In [ ]:
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import matplotlib.cm as cm

def focus_area_percentage(cam, threshold=0.5):
    return float((cam >= threshold).sum() / cam.size)

def cam_entropy(cam):
    cam_flat = cam.flatten()
    cam_flat = cam_flat / (cam_flat.sum() + 1e-8)
    return -np.sum(cam_flat * np.log(cam_flat + 1e-8))

# Sample from both sets
internal_sample = internal_results_df.sample(30, random_state=42)
external_sample = external_results_df.sample(min(30, len(external_results_df)), random_state=42)

xai_comparison = []

print("Computing XAI metrics on both sets...")

for dataset_name, sample_df in [('Internal', internal_sample), ('External', external_sample)]:
    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc=dataset_name):
        try:
            # Get original image
            if dataset_name == 'Internal':
                img_row = internal_df[internal_df['image_id'] == row['image_id']].iloc[0]
                orig = np.array(Image.open(img_row['filepath']).convert('RGB').resize((224, 224)))
            else:
                img_row = external_df[external_df['image_id'] == row['image_id']].iloc[0]
                img_path = Path(f'../Skin_Lesion_Classification_backend/ml/data/raw/isic2020/{row["image_id"]}.jpg')
                if not img_path.exists():
                    continue
                orig = np.array(Image.open(img_path).convert('RGB').resize((224, 224)))
            
            input_t = transform(image=orig)['image'].unsqueeze(0).to(device)
            
            with torch.no_grad():
                out = model(input_t)
                probs = torch.softmax(out, dim=1)[0]
                pred = probs.argmax().item()

            with GradCAM(model=model, target_layers=[target_layer]) as cam_gen:
                cam = cam_gen(input_tensor=input_t, targets=[ClassifierOutputTarget(pred)])[0]

            xai_comparison.append({
                'dataset': dataset_name,
                'image_id': row['image_id'],
                'correct': row['correct'],
                'confidence': row['confidence'],
                'fap_05': focus_area_percentage(cam, 0.5),
                'entropy': cam_entropy(cam),
                'cam_max': cam.max(),
                'cam_mean': cam.mean()
            })
        except Exception as e:
            pass

xai_df = pd.DataFrame(xai_comparison)
xai_df.to_csv('../outputs/metrics/RQ6_xai_external.csv', index=False)
print(f"\nXAI metrics computed: {len(xai_df)} images")

---

## CELL 8: XAI Comparison — Internal vs External

In [ ]:
import seaborn as sns

print("=== XAI Metrics: Internal vs External ===\n")
print(xai_df.groupby('dataset')[['fap_05', 'entropy', 'cam_max', 'cam_mean', 'confidence']].mean().round(4))

# Statistical tests
from scipy.stats import mannwhitneyu

for metric in ['fap_05', 'entropy', 'confidence']:
    int_vals = xai_df[xai_df['dataset'] == 'Internal'][metric]
    ext_vals = xai_df[xai_df['dataset'] == 'External'][metric]
    stat, p = mannwhitneyu(int_vals, ext_vals)
    sig = '\u2705' if p < 0.05 else '\u274c'
    print(f"\n{metric}: Internal={int_vals.mean():.4f}, External={ext_vals.mean():.4f}, p={p:.4f} {sig}")

# Visualize
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

metrics = ['fap_05', 'entropy', 'cam_max', 'confidence']
titles = ['Focus Area %\n(lower = more focused)', 'Entropy\n(lower = more concentrated)',
           'CAM Max\n(highest activation)', 'Model Confidence']

for ax, metric, title in zip(axes.flatten(), metrics, titles):
    sns.boxplot(data=xai_df, x='dataset', y=metric, ax=ax, palette=['steelblue', 'coral'])
    ax.set_title(title)
    ax.set_xlabel('')

plt.suptitle('XAI Metrics: Internal (HAM10000) vs External (ISIC 2020)', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/RQ6_xai_comparison.png', dpi=150)
plt.show()

---

## CELL 9: RQ4 Integration — Does Disagreement Signal OOD?

**Key question**: Does the RQ4 insight (inter-method disagreement predicts errors)
also work for distribution shift? If external images show higher disagreement,
disagreement could be a real-time OOD detection signal.

In [ ]:
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, EigenCAM

cam_methods = {'GradCAM': GradCAM, 'GradCAM++': GradCAMPlusPlus, 'EigenCAM': EigenCAM}

def jaccard_similarity(cam1, cam2, threshold=0.5):
    b1 = (cam1 >= threshold).astype(bool)
    b2 = (cam2 >= threshold).astype(bool)
    intersection = np.logical_and(b1, b2).sum()
    union = np.logical_or(b1, b2).sum()
    return float(intersection / union) if union > 0 else 0.0

disagreement_results = []

print("Computing inter-method disagreement on both sets...")

for dataset_name, sample_df in [('Internal', internal_sample), ('External', external_sample)]:
    for _, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc=dataset_name):
        try:
            if dataset_name == 'Internal':
                img_row = internal_df[internal_df['image_id'] == row['image_id']].iloc[0]
                orig = np.array(Image.open(img_row['filepath']).convert('RGB').resize((224, 224)))
            else:
                img_row = external_df[external_df['image_id'] == row['image_id']].iloc[0]
                img_path = Path(f'../Skin_Lesion_Classification_backend/ml/data/raw/isic2020/{row["image_id"]}.jpg')
                if not img_path.exists():
                    continue
                orig = np.array(Image.open(img_path).convert('RGB').resize((224, 224)))
            
            input_t = transform(image=orig)['image'].unsqueeze(0).to(device)
            
            with torch.no_grad():
                out = model(input_t)
                pred = torch.softmax(out, dim=1)[0].argmax().item()

            cams = {}
            for name, cam_class in cam_methods.items():
                with cam_class(model=model, target_layers=[target_layer]) as cam_gen:
                    cams[name] = cam_gen(input_tensor=input_t, targets=[ClassifierOutputTarget(pred)])[0]

            # Pairwise Jaccard
            jaccards = []
            method_names = list(cams.keys())
            for i in range(len(method_names)):
                for j in range(i+1, len(method_names)):
                    jaccards.append(jaccard_similarity(cams[method_names[i]], cams[method_names[j]]))
            
            disagreement_results.append({
                'dataset': dataset_name,
                'image_id': row['image_id'],
                'avg_jaccard': np.mean(jaccards),
                'correct': row['correct']
            })
        except:
            pass

disagreement_df = pd.DataFrame(disagreement_results)

print("\n=== Inter-method Disagreement ===")
print(disagreement_df.groupby('dataset')[['avg_jaccard', 'correct']].mean().round(4))

# Is disagreement significantly higher on external?
int_jacc = disagreement_df[disagreement_df['dataset'] == 'Internal']['avg_jaccard']
ext_jacc = disagreement_df[disagreement_df['dataset'] == 'External']['avg_jaccard']
stat, p = mannwhitneyu(ext_jacc, int_jacc, alternative='less')
print(f"\nExternal has LOWER agreement (more disagreement): p={p:.4f} {'\u2705 supports' if p < 0.05 else '\u274c not significant'}")

---

## CELL 10: Summary — Does Disagreement Signal OOD?

**Key finding**: If external data shows significantly lower inter-method agreement,
disagreement (RQ4) could be used as a real-time OOD detection signal.
High disagreement → model is on unfamiliar data → flag for human review.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Disagreement box plot
sns.boxplot(data=disagreement_df, x='dataset', y='avg_jaccard', ax=axes[0], palette=['steelblue', 'coral'])
axes[0].set_title('Inter-method Agreement\n(Internal vs External)')
axes[0].set_ylabel('Avg Jaccard Similarity\n(higher = methods agree)')
axes[0].set_xlabel('')

# Disagreement vs correctness on external
ext_dis = disagreement_df[disagreement_df['dataset'] == 'External']
correct = ext_dis[ext_dis['correct'] == 1]['avg_jaccard']
incorrect = ext_dis[ext_dis['correct'] == 0]['avg_jaccard']

data_plot = pd.DataFrame({
    'group': ['Correct']*len(correct) + ['Incorrect']*len(incorrect),
    'agreement': pd.concat([correct, incorrect]).values
})
sns.boxplot(data=data_plot, x='group', y='agreement', ax=axes[1], palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('External Only: Agreement vs Correctness')
axes[1].set_ylabel('Avg Jaccard Similarity')
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig('../outputs/figures/RQ6_disagreement_ood.png', dpi=150)
plt.show()

print("\n=== RQ6 SUMMARY ===")
print(f"Performance drop (AUC): {int_auc - ext_auc:.4f}")
print(f"Confidence drop:        {int_avg_conf - ext_avg_conf:.4f}")
print(f"XAI focus change:      {xai_df[xai_df['dataset']=='Internal']['fap_05'].mean() - xai_df[xai_df['dataset']=='External']['fap_05'].mean():.4f}")
print()
print("KEY INSIGHT:")
if p < 0.05 and ext_jacc.mean() < int_jacc.mean():
    print("  \u2705 External data shows HIGHER disagreement — RQ4 disagreement signal works for OOD detection")
    print("  → Deploy disagreement threshold as real-time OOD flag in production")
else:
    print("  Disagreement is NOT a reliable OOD signal for this model/dataset combination")
    print("  → More research needed on uncertainty quantification for this model")